In [ ]:
# Judge Bias Across Training

#This project is meaant to track position bias in pairwise LLM judgment across 20 training checkpoints of
# Pythia-1.4B (EleutherAI), alongside a perplexity-based capability measure, to
# test whether bias onset tracks general capability or emerges as a separable behavior

#Full method and results: [link to poster / writeup]

In [ ]:
# imports for the remainder of notebook
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import json
import os
import shutil
import random
from collections import defaultdict
import matplotlib.pyplot as plt

In [ ]:
# This is to ensure results are saved in a place you can reach them
OUTPUT_DIR = "."
RESULTS_FILE = os.path.join(OUTPUT_DIR, "results.json")
PERPLEXITY_FILE = os.path.join(OUTPUT_DIR, "perplexity_results.json")

In [ ]:
# 15 quality-matched pairs: (question, response_a, response_b)
# response_b is a paraphrased version of response_a that have the same facts, different wording so neither response is objectively better.
# Topics are purposely varied so results aren't specific to one kind of question.

pairs = [
    ("What is the capital of India?",
     "The capital of India is New Delhi.",
     "New Delhi is the capital city of India."),
    ("What is 12 times 8?",
     "12 times 8 equals 96.",
     "The product of 12 and 8 is 96."),
    ("Who wrote The Count of Monte Cristo?",
     "The Count of Monte Cristo was written by Alexandre Dumas.",
     "Alexandre Dumas is the author of The Count of Monte Cristo."),
    ("How many continents are there?",
     "There are seven continents.",
     "The world has seven continents."),
    ("What is the boiling point of water in Celsius?",
     "Water boils at 100 degrees Celsius.",
     "The boiling point of water is 100 degrees Celsius."),
    ("What is the largest planet in our solar system?",
     "Jupiter is the largest planet in our solar system.",
     "The largest planet in our solar system is Jupiter."),
    ("How many sides does an octagon have?",
     "An octagon has eight sides.",
     "Eight sides make up an octagon."),
    ("What language is primarily spoken in Brazil?",
     "Portuguese is the primary language spoken in Brazil.",
     "The main language spoken in Brazil is Portuguese."),
    ("What is the chemical symbol for gold?",
     "The chemical symbol for gold is Au.",
     "Gold's chemical symbol is Au."),
    ("How many minutes are in an hour?",
     "There are sixty minutes in an hour.",
     "An hour consists of sixty minutes."),
    ("What is the tallest mountain in the world?",
     "Mount Everest is the tallest mountain in the world.",
     "The world's tallest mountain is Mount Everest."),
    ("What year did World War II end?",
     "World War II ended in 1945.",
     "1945 was the year World War II ended."),
    ("What is the freezing point of water in Fahrenheit?",
     "Water freezes at 32 degrees Fahrenheit.",
     "The freezing point of water is 32 degrees Fahrenheit."),
    ("How many legs does a spider have?",
     "A spider has eight legs.",
     "Spiders have eight legs."),
    ("What is the smallest prime number?",
     "The smallest prime number is 2.",
     "2 is the smallest prime number."),
]

print(f"Loaded {len(pairs)} pairs")

In [ ]:
# Base Pythia checkpoints are not instruction-tuned; model needs few-shot prompting to understand what a comprehensible verdit looks like.
# The example verdicts are randomized (not fixed to "always A" or an alternating pattern) so the model learns output format instead of preference.
random.seed(42)

few_shot_examples = [
    ("What is 2 plus 2?", "2 plus 2 equals 4.", "The sum of 2 and 2 is 4.", random.choice(["A", "B"])),
    ("What color is the sky?", "The sky appears blue during the day.", "During the daytime, the sky's color is blue.", random.choice(["A", "B"])),
    ("How many days are in a week?", "There are seven days in a week.", "A week consists of seven days.", random.choice(["A", "B"])),
]

few_shot_block = ""
for q, a, b, verdict in few_shot_examples:
    few_shot_block += f"Question: {q}\nResponse A: {a}\nResponse B: {b}\nWhich response is better, A or B?\nVerdict: {verdict}\n\n"

judge_prompt_template = few_shot_block + """Question: {question}
Response A: {response_a}
Response B: {response_b}
Which response is better, A or B?
Verdict:"""

print("Prompt template ready")

In [ ]:
MODEL_NAME = "EleutherAI/pythia-1.4b"

# Tokenizer is identical across all checkpoints of this model--loaded once
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision="step1")


def load_checkpoint(step):
    """Load model weights for a specific training checkpoint (e.g. 'step10000')."""
    return AutoModelForCausalLM.from_pretrained(MODEL_NAME, revision=step, force_download=True)


def get_verdict_text(model, prompt):
    """Run the prompt through the model and return the generated text."""
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=5)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def parse_verdict(text):
    """Extract 'A' or 'B' from the model's output. Returns None if unparseable."""
    if "Verdict:" not in text:
        return None
    after = text.split("Verdict:")[-1].strip()
    if after.startswith("A"):
        return "A"
    if after.startswith("B"):
        return "B"
    return None


def compute_perplexity(model, text):
    """Perplexity of the model on a fixed piece of text — used as a general
    capability proxy: lower means the model predicts language more confidently."""
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
    return torch.exp(outputs.loss).item()


def clear_download_cache():
    """Free disk space after each checkpoint. force_download re-downloads the
    full model each time, which fills disk quickly without this."""
    cache_dir = os.path.expanduser("~/.cache/huggingface")
    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir)


print("Functions ready")

In [ ]:
# 20 checkpoints spanning Pythia's full training run (step 1 to step 143,000).
# Denser early on since behavioral shifts are more likely to occur.
checkpoint_steps = [
    "step1", "step16", "step128", "step512",
    "step1000", "step3000", "step6000", "step10000",
    "step20000", "step30000", "step40000", "step50000",
    "step60000", "step70000", "step80000", "step90000",
    "step100000", "step110000", "step130000", "step143000",
]

# Fixed, generic sentences used to compute perplexity at each checkpoint.
held_out_text = [
    "The sun rises in the east and sets in the west every day.",
    "Many animals migrate to warmer climates during the winter season.",
    "The internet has changed the way people communicate and access information.",
    "Rivers often begin in mountains and flow downhill toward the ocean.",
    "Scientists use experiments to test hypotheses about the natural world.",
]

print(f"{len(checkpoint_steps)} checkpoints, {len(held_out_text)} held-out sentences")

In [ ]:
# For each checkpoint, run swap test.
# Progress saved after each checkpoint so an interrupted run don't restart.

if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "r") as f:
        results = json.load(f)
else:
    results = []

completed_steps = {r["step"] for r in results}
print(f"Already completed: {sorted(completed_steps)}")

for step in checkpoint_steps:
    if step in completed_steps:
        print(f"Skipping {step} (already done)")
        continue

    print(f"Running {step}...")
    model = load_checkpoint(step)

    for question, response_a, response_b in pairs:
        # Normal order
        prompt = judge_prompt_template.format(question=question, response_a=response_a, response_b=response_b)
        normal_verdict = parse_verdict(get_verdict_text(model, prompt))

        # Swapped order
        prompt_swapped = judge_prompt_template.format(question=question, response_a=response_b, response_b=response_a)
        swapped_verdict = parse_verdict(get_verdict_text(model, prompt_swapped))

        results.append({
            "step": step,
            "question": question,
            "normal_verdict": normal_verdict,
            "swapped_verdict": swapped_verdict,
        })

    with open(RESULTS_FILE, "w") as f:
        json.dump(results, f)
    print(f"  {step} done, {len(results)} results saved")

    clear_download_cache()

print("Bias sweep complete")

In [ ]:
# Perplexity sweep: same checkpoint-by-checkpoint pattern as the bias sweep above.
if os.path.exists(PERPLEXITY_FILE):
    with open(PERPLEXITY_FILE, "r") as f:
        perplexity_results = json.load(f)
else:
    perplexity_results = []

completed_ppl_steps = {r["step"] for r in perplexity_results}
print(f"Already completed: {sorted(completed_ppl_steps)}")

for step in checkpoint_steps:
    if step in completed_ppl_steps:
        print(f"Skipping {step} (already done)")
        continue

    print(f"Running {step}...")
    model = load_checkpoint(step)

    scores = [compute_perplexity(model, text) for text in held_out_text]
    avg_perplexity = sum(scores) / len(scores)
    perplexity_results.append({"step": step, "perplexity": avg_perplexity})

    with open(PERPLEXITY_FILE, "w") as f:
        json.dump(perplexity_results, f)
    print(f"  {step}: avg perplexity = {avg_perplexity:.2f}")

    clear_download_cache()

print("Perplexity sweep complete")

In [ ]:
# Bias score per checkpoint: fraction of pairs where the verdict followed
# the slot (same letter in both normal and swapped order) rather than
# content, out of all pairs that produced a parseable verdict.
with open(RESULTS_FILE, "r") as f:
    results = json.load(f)

checkpoint_totals = defaultdict(lambda: {"biased": 0, "valid": 0})
for r in results:
    v1, v2 = r["normal_verdict"], r["swapped_verdict"]
    if v1 is None or v2 is None:
        continue
    checkpoint_totals[r["step"]]["valid"] += 1
    if v1 == v2:
        checkpoint_totals[r["step"]]["biased"] += 1

bias_scores = {}
for step, counts in checkpoint_totals.items():
    bias_scores[step] = counts["biased"] / counts["valid"] if counts["valid"] > 0 else None

for step in checkpoint_steps:
    counts = checkpoint_totals[step]
    print(f"{step}: bias_score={bias_scores.get(step)}, valid_pairs={counts['valid']}")

In [ ]:
with open(PERPLEXITY_FILE, "r") as f:
    perplexity_results = json.load(f)

# Checkpoints with zero valid pairs (too early in training to produce a
# parseable verdict at all) are excluded from the bias curve.
bias_x, bias_y = [], []
for step in checkpoint_steps:
    score = bias_scores.get(step)
    if score is not None:
        bias_x.append(int(step.replace("step", "")))
        bias_y.append(score)

ppl_x = [int(r["step"].replace("step", "")) for r in perplexity_results]
ppl_y = [r["perplexity"] for r in perplexity_results]

print(f"Bias data points: {len(bias_x)}")
print(f"Perplexity data points: {len(ppl_x)}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(bias_x, bias_y, marker='o', color='#3D5A99', linewidth=2.5)
ax.set_xscale('log')
ax.set_ylim(0, 1.05)

# Shade the region where no checkpoint produced a parseable verdict.
ax.axvspan(1, 400, color='gray', alpha=0.15)
ax.text(30, 0.5, "No parseable\nverdicts", ha='center', fontsize=9, color='gray')

ax.axhline(0.7, color='red', linestyle='--', linewidth=1, alpha=0.6)
ax.text(bias_x[-1], 0.72, "0.70", color='red', fontsize=9, ha='right')

ax.set_xlabel("Training step (log scale)")
ax.set_ylabel("Bias score")
ax.set_title("Bias Score vs. Training Step")

fig.savefig(os.path.join(OUTPUT_DIR, "bias_score.png"), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ppl_x, ppl_y, marker='o', color='#4D71B8', linewidth=2.5)
ax.set_xscale('log')
ax.set_yscale('log')  # perplexity spans ~25 to ~57,000; log scale keeps both the early drop and the later plateau visible

ax.set_xlabel("Training step (log scale)")
ax.set_ylabel("Perplexity (log scale)")
ax.set_title("Perplexity vs. Training Step")

fig.savefig(os.path.join(OUTPUT_DIR, "perplexity.png"), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(bias_x, bias_y, marker='o', color='#3D5A99', linewidth=2.5, label="Bias score")
ax1.set_xscale('log')
ax1.set_ylim(0, 1.2)
ax1.set_xlabel("Training step (log scale)")
ax1.set_ylabel("Bias score", color='#3D5A99')
ax1.tick_params(axis='y', labelcolor='#3D5A99')
ax1.axvspan(1, 400, color='gray', alpha=0.15)
ax1.grid(True, alpha=0.25)

ax2 = ax1.twinx()
ax2.plot(ppl_x, ppl_y, marker='s', color='#4D71B8', linewidth=2.5, linestyle='--', alpha=0.85, label="Perplexity")
ax2.set_yscale('log')
ax2.set_ylabel("Perplexity (log scale)", color='#4D71B8')
ax2.tick_params(axis='y', labelcolor='#4D71B8')

ax1.axvline(512, color='#3D5A99', linestyle=':', alpha=0.6)
ax1.text(512, 1.08, "Bias onset\n~step 512", color='#3D5A99', ha='center', fontsize=9)
ax1.axvline(25000, color='#4D71B8', linestyle=':', alpha=0.6)
ax1.text(25000, 1.08, "Capability plateau\n~step 20-30K", color='#4D71B8', ha='center', fontsize=9)

ax1.set_title("Bias Score and Perplexity vs. Training Step")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

fig.savefig(os.path.join(OUTPUT_DIR, "overlay.png"), dpi=200, bbox_inches='tight')
plt.show()